<a href="https://colab.research.google.com/github/Zhalil24/BreastMRI-CNN-Classification/blob/main/ResNet34NewResult.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.model_selection import StratifiedKFold

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import os
import shutil
import copy

from google.colab import drive
from IPython.display import display


# ============================================================
# 2. GPU / CUDA AYARI VE DRIVE BAĞLAMA
# ============================================================

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Sistem: {device} üzerinde çalışıyor.")

drive.mount('/content/drive')


# ============================================================
# 3. DATASET RAR DOSYASINI KOPYALAMA VE AÇMA
# ============================================================

rar_source_path = "/content/drive/MyDrive/data_set_zip/breast_mri_dataset.rar"
rar_target_path = "/content/breast_mri_dataset.rar"

shutil.copy(rar_source_path, rar_target_path)

# Aynı dosyalar varsa üzerine yazsın diye -o+ kullanıldı.
os.system(f'unrar x -o+ "{rar_target_path}" /content/')


# ============================================================
# 4. GENEL AYARLAR
# ============================================================

data_dir = "/content/breast_mri_dataset"

BATCH_SIZE = 32
NUM_EPOCHS = 10
NUM_CLASSES = 2


# ============================================================
# 5. ÇIKTI KLASÖRLERİ
# ============================================================

output_root = "/content/drive/MyDrive/breast_mri_resnet34_outputs"
charts_dir = os.path.join(output_root, "charts")
tables_dir = os.path.join(output_root, "tables")

os.makedirs(output_root, exist_ok=True)
os.makedirs(charts_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

print(f"Çıktı klasörü: {output_root}")


# ============================================================
# 6. VERİ HAZIRLAMA VE AUGMENTATION
# ============================================================

mri_mean = [0.485, 0.456, 0.406]
mri_std = [0.229, 0.224, 0.225]

data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomAffine(
            degrees=0,
            translate=(0.05, 0.05),
            scale=(0.95, 1.05)
        ),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(mri_mean, mri_std)
    ]),

    "val_test": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mri_mean, mri_std)
    ]),
}


# ============================================================
# 7. DATASETLERİN YÜKLENMESİ
# ============================================================

# Eğitim için augmentation'lı train dataset
full_train_val_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "train"),
    transform=data_transforms["train"]
)

# Fold validation için augmentation'sız train dataset
full_train_val_dataset_eval = datasets.ImageFolder(
    os.path.join(data_dir, "train"),
    transform=data_transforms["val_test"]
)

test_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "test"),
    transform=data_transforms["val_test"]
)

labels = np.array(full_train_val_dataset.targets)

class_names = full_train_val_dataset.classes
print("Sınıflar:", full_train_val_dataset.class_to_idx)

if len(class_names) == 2:
    display_class_names = ["Benign", "Malignant"]
else:
    display_class_names = class_names


# ============================================================
# 8. RESNET34 MODEL OLUŞTURMA
# ============================================================

def build_model():
    model = models.resnet34(
        weights=models.ResNet34_Weights.IMAGENET1K_V1
    )

    # conv1, bn1, relu, maxpool ve layer1 dondurulur.
    for name, child in model.named_children():
        if name in ["conv1", "bn1", "relu", "maxpool", "layer1"]:
            for param in child.parameters():
                param.requires_grad = False
        else:
            for param in child.parameters():
                param.requires_grad = True

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

    return model.to(device)


# ============================================================
# 9. DİFERANSİYEL ÖĞRENME ORANI
# ============================================================

def get_optimizer(model):
    optimizer = optim.Adam([
        {
            "params": model.layer2.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.layer3.parameters(),
            "lr": 5e-5
        },
        {
            "params": model.layer4.parameters(),
            "lr": 1e-4
        },
        {
            "params": model.fc.parameters(),
            "lr": 1e-3
        }
    ])

    return optimizer


criterion = nn.CrossEntropyLoss()


# ============================================================
# 10. MODEL AĞIRLIKLARINI GÜVENLİ KOPYALAMA
# ============================================================

def clone_state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


# ============================================================
# 11. EĞİTİM FONKSİYONU
# ============================================================

def train_model(fold_no, train_loader, val_loader, num_epochs=10):
    model = build_model()
    optimizer = get_optimizer(model)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_epoch_val_acc = -1
    best_epoch_no = 1
    best_epoch_state = None

    for epoch in range(num_epochs):
        # -------------------------
        # Eğitim
        # -------------------------
        model.train()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels_batch in train_loader:
            inputs = inputs.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels_batch.data).item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects / len(train_loader.dataset)

        # -------------------------
        # Validation
        # -------------------------
        model.eval()

        val_running_loss = 0.0
        val_running_corrects = 0

        with torch.no_grad():
            for inputs, labels_batch in val_loader:
                inputs = inputs.to(device)
                labels_batch = labels_batch.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)

                loss = criterion(outputs, labels_batch)

                val_running_loss += loss.item() * inputs.size(0)
                val_running_corrects += torch.sum(preds == labels_batch.data).item()

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = val_running_corrects / len(val_loader.dataset)

        history["train_loss"].append(epoch_loss)
        history["train_acc"].append(epoch_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Fold {fold_no} Epoch {epoch+1}/{num_epochs}: "
            f"Train Loss: {epoch_loss:.4f} "
            f"Acc: {epoch_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} "
            f"Acc: {val_acc:.4f}"
        )

        if val_acc > best_epoch_val_acc:
            best_epoch_val_acc = val_acc
            best_epoch_no = epoch + 1
            best_epoch_state = clone_state_dict_to_cpu(model)

    # Fold içindeki en iyi epoch ağırlıkları geri yükleniyor.
    if best_epoch_state is not None:
        model.load_state_dict(best_epoch_state)

    return model, history, best_epoch_no


# ============================================================
# 12. DETAYLI DEĞERLENDİRME FONKSİYONU
# ============================================================

def evaluate_model(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels_batch in loader:
            inputs = inputs.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(inputs)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    metrics = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall": recall_score(all_labels, all_preds, zero_division=0),
        "f1_score": f1_score(all_labels, all_preds, zero_division=0)
    }

    try:
        metrics["roc_auc"] = roc_auc_score(all_labels, all_probs)
    except ValueError:
        metrics["roc_auc"] = np.nan

    return metrics, all_labels, all_preds, all_probs


# ============================================================
# 13. GRAFİK FONKSİYONLARI
# ============================================================

def plot_accuracy(history, fold_no, save_path):
    epochs_range = range(1, len(history["train_acc"]) + 1)

    plt.figure(figsize=(9, 6))
    plt.plot(
        epochs_range,
        history["train_acc"],
        marker="o",
        label="Train Accuracy"
    )
    plt.plot(
        epochs_range,
        history["val_acc"],
        marker="o",
        label="Validation Accuracy"
    )

    plt.title(f"Fold {fold_no}/5 Accuracy Grafiği")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_loss(history, fold_no, save_path):
    epochs_range = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(9, 6))
    plt.plot(
        epochs_range,
        history["train_loss"],
        marker="o",
        label="Train Loss"
    )
    plt.plot(
        epochs_range,
        history["val_loss"],
        marker="o",
        label="Validation Loss"
    )

    plt.title(f"Fold {fold_no}/5 Loss Grafiği")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_confusion_matrix_graph(labels_data, preds_data, title, save_path):
    cm = confusion_matrix(labels_data, preds_data, labels=[0, 1])

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=display_class_names,
        yticklabels=display_class_names
    )

    plt.title(title)
    plt.ylabel("Gerçek")
    plt.xlabel("Tahmin")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_roc_curve_graph(labels_data, probs_data, title, save_path):
    plt.figure(figsize=(8, 6))

    try:
        fpr, tpr, _ = roc_curve(labels_data, probs_data)
        auc_value = roc_auc_score(labels_data, probs_data)

        plt.plot(
            fpr,
            tpr,
            lw=2,
            label=f"ROC Curve - AUC = {auc_value:.4f}"
        )

        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            lw=2,
            label="Random Classifier"
        )

    except ValueError:
        plt.text(
            0.5,
            0.5,
            "ROC hesaplanamadı.\nValidation setinde tek sınıf olabilir.",
            ha="center",
            va="center"
        )

    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


# ============================================================
# 14. 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

all_histories = []
fold_summary_rows = []
epoch_history_rows = []
output_file_rows = []

best_overall_val_acc = -1
best_overall_state = None
best_overall_fold = None

last_fold_state = None

# Final testte validation accuracy değeri en iyi fold kullanılacak.
FINAL_MODEL_POLICY = "best_fold"

print("\n--- 5-Fold Eğitim Başlıyor ---")

for fold_no, (train_idx, val_idx) in enumerate(
    skf.split(np.zeros(len(labels)), labels),
    start=1
):
    print(f"\n--- FOLD {fold_no} BAŞLIYOR ---")

    train_sub = Subset(full_train_val_dataset, train_idx)

    # Validation tarafında augmentation kullanılmıyor.
    val_sub = Subset(full_train_val_dataset_eval, val_idx)

    train_loader = DataLoader(
        train_sub,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_sub,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    model, history, best_epoch_no = train_model(
        fold_no,
        train_loader,
        val_loader,
        num_epochs=NUM_EPOCHS
    )

    all_histories.append(history)

    val_metrics, val_labels, val_preds, val_probs = evaluate_model(
        model,
        val_loader
    )

    print(f"\nFold {fold_no} Validation Performansı")
    print(f"Best Epoch: {best_epoch_no}")
    print(f"Accuracy:  {val_metrics['accuracy']:.4f}")
    print(f"Precision: {val_metrics['precision']:.4f}")
    print(f"Recall:    {val_metrics['recall']:.4f}")
    print(f"F1-Score:  {val_metrics['f1_score']:.4f}")
    print(f"ROC-AUC:   {val_metrics['roc_auc']:.4f}")

    accuracy_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_accuracy.png"
    )

    loss_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_loss.png"
    )

    cm_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_confusion_matrix.png"
    )

    roc_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_roc_curve.png"
    )

    # Her fold için 4 grafik
    plot_accuracy(
        history,
        fold_no,
        accuracy_path
    )

    plot_loss(
        history,
        fold_no,
        loss_path
    )

    plot_confusion_matrix_graph(
        val_labels,
        val_preds,
        title=f"Fold {fold_no}/5 Validation Confusion Matrix",
        save_path=cm_path
    )

    plot_roc_curve_graph(
        val_labels,
        val_probs,
        title=f"Fold {fold_no}/5 Validation ROC-AUC Grafiği",
        save_path=roc_path
    )

    fold_summary_rows.append({
        "Fold": fold_no,
        "Best_Epoch": best_epoch_no,
        "Validation_Accuracy": val_metrics["accuracy"],
        "Validation_Precision": val_metrics["precision"],
        "Validation_Recall": val_metrics["recall"],
        "Validation_F1_Score": val_metrics["f1_score"],
        "Validation_ROC_AUC": val_metrics["roc_auc"],
        "Accuracy_Graph": accuracy_path,
        "Loss_Graph": loss_path,
        "Confusion_Matrix_Graph": cm_path,
        "ROC_Curve_Graph": roc_path
    })

    output_file_rows.extend([
        {
            "Output_Type": "Fold Accuracy Graph",
            "Fold": fold_no,
            "File_Path": accuracy_path
        },
        {
            "Output_Type": "Fold Loss Graph",
            "Fold": fold_no,
            "File_Path": loss_path
        },
        {
            "Output_Type": "Fold Confusion Matrix Graph",
            "Fold": fold_no,
            "File_Path": cm_path
        },
        {
            "Output_Type": "Fold ROC Curve Graph",
            "Fold": fold_no,
            "File_Path": roc_path
        }
    ])

    last_fold_state = clone_state_dict_to_cpu(model)

    if val_metrics["accuracy"] > best_overall_val_acc:
        best_overall_val_acc = val_metrics["accuracy"]
        best_overall_fold = fold_no
        best_overall_state = clone_state_dict_to_cpu(model)

print("\n--- Tüm Foldlar Başarıyla Tamamlandı ---")


# ============================================================
# 15. FOLD TABLOLARI
# ============================================================

for history_index, history_item in enumerate(all_histories, start=1):
    for epoch_index in range(len(history_item["train_acc"])):
        epoch_history_rows.append({
            "Fold": history_index,
            "Epoch": epoch_index + 1,
            "Train_Loss": history_item["train_loss"][epoch_index],
            "Train_Accuracy": history_item["train_acc"][epoch_index],
            "Validation_Loss": history_item["val_loss"][epoch_index],
            "Validation_Accuracy": history_item["val_acc"][epoch_index]
        })

fold_summary_df = pd.DataFrame(fold_summary_rows)
epoch_history_df = pd.DataFrame(epoch_history_rows)

print("\n--- FOLD ÖZET TABLOSU ---")
display(fold_summary_df)

print("\n--- EPOCH GEÇMİŞ TABLOSU ---")
display(epoch_history_df)


# ============================================================
# 16. FINAL TEST SETİ DEĞERLENDİRME
# ============================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

if FINAL_MODEL_POLICY == "last_fold":
    final_model_state = last_fold_state
    final_model_name = "Last Fold Model"
else:
    final_model_state = best_overall_state
    final_model_name = f"Best Fold Model - Fold {best_overall_fold}"

final_model = build_model()
final_model.load_state_dict(final_model_state)
final_model.to(device)

test_metrics, test_labels, test_preds, test_probs = evaluate_model(
    final_model,
    test_loader
)

print("\n--- TEST SETİ PERFORMANS SONUÇLARI ---")
print(f"Final Model: {final_model_name}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1-Score:  {test_metrics['f1_score']:.4f}")
print(f"ROC-AUC:   {test_metrics['roc_auc']:.4f}")

final_test_df = pd.DataFrame([{
    "Final_Model": final_model_name,
    "Test_Accuracy": test_metrics["accuracy"],
    "Test_Precision": test_metrics["precision"],
    "Test_Recall": test_metrics["recall"],
    "Test_F1_Score": test_metrics["f1_score"],
    "Test_ROC_AUC": test_metrics["roc_auc"]
}])

print("\n--- FINAL TEST TABLOSU ---")
display(final_test_df)


# ============================================================
# 17. MEVCUT ORTALAMA ACCURACY GRAFİĞİ
# ============================================================

avg_train_acc = np.mean(
    [h["train_acc"] for h in all_histories],
    axis=0
)

avg_val_acc = np.mean(
    [h["val_acc"] for h in all_histories],
    axis=0
)

avg_train_loss = np.mean(
    [h["train_loss"] for h in all_histories],
    axis=0
)

avg_val_loss = np.mean(
    [h["val_loss"] for h in all_histories],
    axis=0
)

epochs_range = range(1, len(avg_train_acc) + 1)

average_accuracy_path = os.path.join(
    charts_dir,
    "average_training_vs_validation_accuracy.png"
)

plt.figure(figsize=(9, 6))
plt.plot(
    epochs_range,
    avg_train_acc,
    marker="o",
    label="Average Train Acc"
)
plt.plot(
    epochs_range,
    avg_val_acc,
    marker="o",
    label="Average Val Acc"
)

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(average_accuracy_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

output_file_rows.append({
    "Output_Type": "Average Accuracy Graph",
    "Fold": "All",
    "File_Path": average_accuracy_path
})


# ============================================================
# 18. MEVCUT FINAL TEST CONFUSION MATRIX
# ============================================================

final_test_cm_path = os.path.join(
    charts_dir,
    "final_test_confusion_matrix.png"
)

cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=display_class_names,
    yticklabels=display_class_names
)

plt.title("Confusion Matrix")
plt.ylabel("Gerçek")
plt.xlabel("Tahmin")
plt.tight_layout()
plt.savefig(final_test_cm_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

output_file_rows.append({
    "Output_Type": "Final Test Confusion Matrix Graph",
    "Fold": "Final Test",
    "File_Path": final_test_cm_path
})


# ============================================================
# 19. MEVCUT FINAL TEST ROC CURVE
# ============================================================

final_test_roc_path = os.path.join(
    charts_dir,
    "final_test_roc_curve.png"
)

fpr, tpr, _ = roc_curve(test_labels, test_probs)
test_auc = roc_auc_score(test_labels, test_probs)

plt.figure(figsize=(7, 5))
plt.plot(
    fpr,
    tpr,
    lw=2,
    label=f"ROC curve area = {test_auc:.2f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    lw=2,
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig(final_test_roc_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

output_file_rows.append({
    "Output_Type": "Final Test ROC Curve Graph",
    "Fold": "Final Test",
    "File_Path": final_test_roc_path
})


# ============================================================
# 20. TÜM SONUÇLARI EXCEL DOSYASINA KAYDETME
# ============================================================

output_files_df = pd.DataFrame(output_file_rows)

excel_path = os.path.join(
    tables_dir,
    "resnet34_breast_mri_model_results.xlsx"
)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    fold_summary_df.to_excel(
        writer,
        sheet_name="Fold_Summary",
        index=False
    )

    epoch_history_df.to_excel(
        writer,
        sheet_name="Epoch_History",
        index=False
    )

    final_test_df.to_excel(
        writer,
        sheet_name="Final_Test",
        index=False
    )

    output_files_df.to_excel(
        writer,
        sheet_name="Output_Files",
        index=False
    )

print("\n--- KAYIT İŞLEMİ TAMAMLANDI ---")
print(f"Grafikler klasörü: {charts_dir}")
print(f"Excel dosyası: {excel_path}")

print("\nÜretilen çıktı dosyaları:")
display(output_files_df)